# 📄 Notebook 2: Arabic Text Summarization

**Project:** Deep Learning Based Arabic Audio Understanding and Retrieval System  
**Task:** Neural Text Summarization  
**Model:** Qwen2.5-1.5B-Instruct (instruction-tuned LLM with Arabic support)  
**Dataset:** Transcripts from Notebook 1 + XL-Sum Arabic (test split)  
**Metrics:** ROUGE-1, ROUGE-2, ROUGE-L  

**Expected Runtime:** 10-15 minutes on Kaggle T4 (200 transcripts + 20 XL-Sum eval)  
**Expected ROUGE-1:** 0.20-0.35 on XL-Sum Arabic test set

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers sentencepiece datasets rouge-score accelerate torch tqdm
print("✅ Installation complete.")

In [ ]:
# Cell 2: Imports & GPU Check
import torch
import time
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

print("=" * 60)
print("GPU / SYSTEM CHECK")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = "cuda"
else:
    print("WARNING: No GPU detected.")
    device = "cpu"
print("=" * 60)

## 1. Model Loading

We use **Qwen/Qwen2.5-1.5B-Instruct**, an instruction-tuned language model with excellent Arabic support.

Unlike mT5 (which needs long news articles), this model uses **prompt-based generation** — it can summarize text of any length, including short sentences from speech transcripts.

This approach mirrors the project's Ollama-based architecture but runs directly on the Kaggle GPU.

In [ ]:
# Cell 3: Load Qwen2.5-1.5B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading model: {MODEL_NAME} ...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

load_time = time.time() - t0
print(f"✅ Model loaded in {load_time:.1f}s")
print(f"   Device: {device}")
print(f"   Model size: ~1.5B parameters")

In [ ]:
# Cell 4: Define summarization function (mirrors friend's llm.py prompt)

def summarize_text(text, max_new_tokens=200):
    """
    Summarize Arabic text using instruction-tuned LLM.
    Uses the same Arabic prompt template as the project's llm.py.
    """
    if not text or len(text.strip()) < 5:
        return "(نص قصير جداً)"
    
    # Count words
    word_count = len(text.strip().split())
    
    # For very short texts (< 15 words), return as-is with tag
    if word_count < 15:
        return f"[جملة قصيرة — {word_count} كلمة] {text.strip()}"
    
    # Build prompt (same as friend's llm.py)
    messages = [
        {"role": "system", "content": "أنت نظام ذكاء اصطناعي متخصص في تلخيص النصوص العربية."},
        {"role": "user", "content": f"قم بتلخيص النص التالي في فقرة واحدة موجزة وواضحة باللغة العربية:\n\nالنص:\n{text}\n\nالملخص:"}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the new tokens
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    summary = tokenizer.decode(generated, skip_special_tokens=True).strip()
    
    return summary if summary else "(فشل التلخيص)"

# Sanity check
test_text = "الذكاء الاصطناعي هو فرع من علوم الحاسوب يهتم بإنشاء أنظمة قادرة على أداء مهام تتطلب ذكاءً بشرياً مثل التعلم والتخطيط وفهم اللغة الطبيعية والتعرف على الأنماط"
test_summary = summarize_text(test_text)
print(f"Sanity check:")
print(f"  Input:   {test_text[:100]}...")
print(f"  Summary: {test_summary[:200]}")

## 2. Summarize Transcripts from Notebook 1

We load the `transcripts.csv` produced by Notebook 1 and generate summaries for each transcript.

**Note:** Upload `transcripts.csv` as a Kaggle dataset first, then update the path below.

In [ ]:
# Cell 5: Load transcripts and summarize

# === UPDATE THIS PATH ===
# Option A: If you uploaded transcripts.csv as a Kaggle dataset:
TRANSCRIPTS_PATH = "/kaggle/input/transcripts-csv/transcripts.csv"
# Option B: If it's in working directory from a previous run:
# TRANSCRIPTS_PATH = "/kaggle/working/transcripts.csv"

if not os.path.exists(TRANSCRIPTS_PATH):
    raise FileNotFoundError(
        f"transcripts.csv not found at {TRANSCRIPTS_PATH}. "
        "Upload it from Notebook 1 outputs as a Kaggle dataset."
    )

df_transcripts = pd.read_csv(TRANSCRIPTS_PATH, encoding="utf-8-sig")
print(f"Loaded {len(df_transcripts)} transcripts.")
print(f"Columns: {list(df_transcripts.columns)}")

# Use 'prediction' column (Whisper output) as input
input_col = "prediction" if "prediction" in df_transcripts.columns else "reference"
print(f"Using column '{input_col}' as summarization input.")

# Summarize all transcripts
MAX_TO_SUMMARIZE = len(df_transcripts)  # Process all
summaries = []

for idx, row in tqdm(df_transcripts.head(MAX_TO_SUMMARIZE).iterrows(), 
                     total=min(MAX_TO_SUMMARIZE, len(df_transcripts)), 
                     desc="Summarizing"):
    text = str(row[input_col]) if pd.notna(row[input_col]) else ""
    if len(text.strip()) < 3:
        summaries.append("(too short)")
        continue
    try:
        s = summarize_text(text)
        summaries.append(s)
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        summaries.append("(error)")

df_transcripts["summary"] = summaries

# Show examples
print("\n" + "=" * 60)
print("SAMPLE OUTPUTS: Original → Summary")
print("=" * 60)
for i in range(min(5, len(df_transcripts))):
    row = df_transcripts.iloc[i]
    print(f"\n--- Sample {i+1} ---")
    print(f"Original: {str(row[input_col])[:200]}")
    print(f"Summary : {str(row['summary'])[:200]}")

## 3. ROUGE Evaluation on XL-Sum Arabic

To obtain an objective quality score, we evaluate the summarization model on the **XL-Sum Arabic** test set.

XL-Sum contains professionally written summaries of BBC Arabic news articles.

**ROUGE Metrics:**
- **ROUGE-1:** Overlap of unigrams (single words)
- **ROUGE-2:** Overlap of bigrams (word pairs)
- **ROUGE-L:** Longest common subsequence (captures sentence structure)

In [ ]:
# Cell 6: ROUGE evaluation on XL-Sum Arabic
NUM_EVAL_SAMPLES = 20

print("Loading XL-Sum Arabic test set...")
try:
    xlsum = load_dataset("csebuetnlp/xlsum", "arabic", split="test", trust_remote_code=True)
    print(f"XL-Sum loaded: {len(xlsum)} total samples.")
except Exception as e:
    print(f"Failed to load XL-Sum: {e}")
    print("Attempting fallback...")
    xlsum = load_dataset("csebuetnlp/xlsum", "arabic", split="test[:100]", trust_remote_code=True)

eval_samples = [xlsum[i] for i in range(min(NUM_EVAL_SAMPLES, len(xlsum)))]
print(f"Evaluating on {len(eval_samples)} XL-Sum samples.")

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
eval_results = []

for idx, sample in enumerate(tqdm(eval_samples, desc="ROUGE evaluation")):
    document = sample.get("text", "")
    reference_summary = sample.get("summary", "")
    
    if not document or not reference_summary:
        continue
    
    try:
        pred_summary = summarize_text(document, max_new_tokens=200)
    except Exception as e:
        print(f"Error on sample {idx}: {e}")
        continue
    
    scores = scorer.score(reference_summary, pred_summary)
    
    r1 = scores["rouge1"].fmeasure
    r2 = scores["rouge2"].fmeasure
    rL = scores["rougeL"].fmeasure
    
    rouge1_scores.append(r1)
    rouge2_scores.append(r2)
    rougeL_scores.append(rL)
    
    eval_results.append({
        "id": idx,
        "document_preview": document[:100] + "...",
        "reference": reference_summary,
        "prediction": pred_summary,
        "rouge1": r1,
        "rouge2": r2,
        "rougeL": rL
    })

mean_r1 = np.mean(rouge1_scores) if rouge1_scores else 0
mean_r2 = np.mean(rouge2_scores) if rouge2_scores else 0
mean_rL = np.mean(rougeL_scores) if rougeL_scores else 0

print("\n" + "=" * 60)
print("ROUGE EVALUATION RESULTS (XL-Sum Arabic)")
print("=" * 60)
print(f"Samples evaluated: {len(rouge1_scores)}")
print(f"ROUGE-1 (F1)    : {mean_r1:.4f}")
print(f"ROUGE-2 (F1)    : {mean_r2:.4f}")
print(f"ROUGE-L (F1)    : {mean_rL:.4f}")
print("=" * 60)

df_eval = pd.DataFrame(eval_results)
print("\nPer-sample ROUGE scores:")
display(df_eval[["id", "rouge1", "rouge2", "rougeL"]].head(10))

In [ ]:
# Cell 7: Qualitative analysis — best and worst summaries
if len(df_eval) == 0:
    print("No evaluation results available.")
else:
    df_sorted = df_eval.sort_values("rougeL", ascending=False).reset_index(drop=True)
    
    print("=" * 60)
    print("TOP 3 BEST SUMMARIES (highest ROUGE-L)")
    print("=" * 60)
    for i in range(min(3, len(df_sorted))):
        row = df_sorted.iloc[i]
        print(f"\n--- Rank {i+1} | ROUGE-L: {row['rougeL']:.3f} ---")
        print(f"Reference : {row['reference'][:250]}")
        print(f"Prediction: {row['prediction'][:250]}")
    
    print("\n" + "=" * 60)
    print("BOTTOM 2 WORST SUMMARIES (lowest ROUGE-L)")
    print("=" * 60)
    for i in range(1, min(3, len(df_sorted)+1)):
        row = df_sorted.iloc[-i]
        print(f"\n--- Rank {len(df_sorted)-i+1} | ROUGE-L: {row['rougeL']:.3f} ---")
        print(f"Reference : {row['reference'][:250]}")
        print(f"Prediction: {row['prediction'][:250]}")

In [ ]:
# Cell 8: Save results
out_csv = "/kaggle/working/summaries.csv"
df_transcripts.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"💾 Saved transcript summaries: {out_csv}")

eval_csv = "/kaggle/working/summarization_eval.csv"
df_eval.to_csv(eval_csv, index=False, encoding="utf-8-sig")
print(f"💾 Saved evaluation results: {eval_csv}")

print("\n" + "=" * 60)
print("📋 RESULTS TABLE FOR REPORT")
print("=" * 60)
print("| Metric    | Value |")
print("|-----------|-------|")
print(f"| Model     | Qwen2.5-1.5B-Instruct |")
print(f"| ROUGE-1   | {mean_r1:.4f} |")
print(f"| ROUGE-2   | {mean_r2:.4f} |")
print(f"| ROUGE-L   | {mean_rL:.4f} |")
print(f"| Eval Samples | {len(rouge1_scores)} |")
print(f"| Transcripts Summarized | {len(df_transcripts)} |")
print("=" * 60)

In [ ]:
# Cell 9: Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: ROUGE score bars
ax1 = axes[0]
metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
means = [mean_r1, mean_r2, mean_rL]
colors = ["#3498db", "#2ecc71", "#9b59b6"]
bars = ax1.bar(metrics, means, color=colors, edgecolor="black", alpha=0.85)
ax1.set_ylabel("F1 Score", fontsize=11)
ax1.set_title("Mean ROUGE Scores (XL-Sum Arabic)", fontsize=13, fontweight="bold")
ax1.set_ylim(0, max(means) * 1.3 if max(means) > 0 else 0.5)
for bar, val in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax1.grid(axis="y", alpha=0.3)

# Plot 2: ROUGE-L distribution
ax2 = axes[1]
if len(rougeL_scores) > 0:
    ax2.hist(rougeL_scores, bins=12, color="#e74c3c", edgecolor="black", alpha=0.8)
    ax2.axvline(x=mean_rL, color="navy", linestyle="--", linewidth=2,
                label=f"Mean = {mean_rL:.3f}")
    ax2.set_xlabel("ROUGE-L F1 Score", fontsize=11)
    ax2.set_ylabel("Frequency", fontsize=11)
    ax2.set_title("ROUGE-L Distribution Across Samples", fontsize=13, fontweight="bold")
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3)
else:
    ax2.text(0.5, 0.5, "No data", ha="center", va="center")

plt.tight_layout()
plt.savefig("/kaggle/working/rouge_analysis.png", dpi=200, bbox_inches="tight")
plt.show()
print("💾 Plot saved: /kaggle/working/rouge_analysis.png")

## Related Work

1. **Qwen2.5:** Qwen Team (2024). "Qwen2.5: A Party of Foundation Models." Alibaba Cloud. A family of large language models supporting 29+ languages including Arabic, with strong instruction-following capabilities.

2. **XL-Sum:** Hasan, T., et al. (2021). "XL-Sum: Large-Scale Multilingual Abstractive Summarization for 44 Languages." *ACL-Findings 2021.* The dataset used for Arabic summarization evaluation.

3. **mT5:** Xue, L., et al. (2021). "mT5: A massively multilingual pre-trained text-to-text transformer." *NAACL 2021.* An alternative approach for multilingual summarization.

### Known Limitations
- **Short inputs:** Very short transcripts (under 15 words) are returned as-is since summarization would not add value.
- **Domain mismatch:** The model is general-purpose; spoken transcripts may differ from training distribution.
- **Hallucination:** LLMs may occasionally generate content not present in the source text.

## Results Summary

| Item | Value |
|------|-------|
| Model | Qwen2.5-1.5B-Instruct |
| Transcripts summarized | *(fill after running)* |
| XL-Sum eval samples | 20 |
| **ROUGE-1 (F1)** | ***(fill after running)*** |
| **ROUGE-2 (F1)** | ***(fill after running)*** |
| **ROUGE-L (F1)** | ***(fill after running)*** |
| GPU Used | Tesla T4 |